# 08. Ridge com features originais das tabelas-fonte

**Objetivo:** montar o dataset de treino **somente com colunas originais** das tabelas-fonte, ligadas aos contratos selecionados pela chave correspondente, e rodar Ridge contra um baseline.

**Diferença para o notebook 07:** lá cada tabela auxiliar era resumida por contrato (contagens e somas), o que cria features novas (por exemplo, `adp_qtd`). Aqui **nenhuma coluna é agregada ou calculada**: cada linha do dataset carrega os valores originais de um registro das tabelas.

## Como o dataset é montado

As tabelas auxiliares têm granularidades diferentes, e cada uma tem várias linhas por contrato:

| Fonte | Linha = | Chave para o contrato |
|---|---|---|
| `vw_modelagem_contratos_filtros` | contrato | `idContrato` |
| `vw_tparcela_liberacao` + `vw_bi_parcelas` | liberação de uma parcela | `id_Contrato`; parcelas ligadas por `id_Contrato` + `id_Parcela` |
| `vw_taditivoprazo` | aditivo de prazo | `idContrato` |
| `vw_taditivovalor` | aditivo de valor | `idContrato` |

Todas entram por `left join` a partir dos contratos. Como cada fonte tem várias linhas por contrato, o join completo pela chave do contrato **multiplica as linhas** (cada liberação é combinada com cada aditivo do mesmo contrato). O alvo é por contrato, então:

- o alvo se repete em todas as linhas do contrato;
- cada linha recebe peso `1 / nº de linhas do contrato`, para todos os contratos contarem igual no treino;
- a previsão de um contrato é a **média das previsões de suas linhas**.

## Alvo, modelos e validação

- **Alvo:** `max(0, duração real − prazo ajustado)`, igual aos notebooks 05 a 07 (duração real = `data_recebimento_provisorio − data_ordem_servico`; o prazo ajustado inclui os aditivos de prazo).
- **Baseline:** mediana do atraso no treino. **Modelo:** Ridge.
- **Validação:** 4 janelas temporais expanding (20→5, 25→5, 30→5, 35→5 contratos) e teste final nos 8 contratos mais recentes, ordenados por `data_recebimento_provisorio`. A separação é por contrato: todas as linhas de um contrato ficam do mesmo lado.

> **Atenção:** liberações e aditivos só existem durante ou depois da obra. O notebook mede quanto esses dados explicam o atraso; não é uma previsão feita na ordem de serviço.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

N_TEST = 8
MIN_TRAIN = 20
BLOCK = 5
N_FOLDS = 4
RIDGE_ALPHA = 1.0

# Se os CSVs estiverem em outra pasta, defina DATA_DIR manualmente.
DATA_DIR = None

## 1. Carga dos dados

In [2]:
FILES = {
    "contratos": "vw_modelagem_contratos_filtros.csv",
    "aditivos_prazo": "vw_taditivoprazo.csv",
    "aditivos_valor": "vw_taditivovalor.csv",
    "liberacoes": "vw_tparcela_liberacao.csv",
    "parcelas": "vw_bi_parcelas.csv",
}


def find_data_dir():
    if DATA_DIR is not None:
        return Path(DATA_DIR)
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "data", cwd / "data" / "raw", Path("/mnt/user-data/uploads")]
    for parent in cwd.parents:
        candidates += [parent / "data", parent / "data" / "raw"]
    for folder in candidates:
        if (folder / FILES["contratos"]).exists():
            return folder
    raise FileNotFoundError("CSVs não encontrados; defina DATA_DIR.")


base = find_data_dir()


def read(name, **kw):
    return pd.read_csv(base / FILES[name], encoding="utf-8-sig", **kw)


contratos = read("contratos", parse_dates=["data_ordem_servico", "data_recebimento_provisorio"])
aditivos_prazo = read("aditivos_prazo", parse_dates=["Data_termino_anterior", "Data_termino_nova"])
aditivos_valor = read("aditivos_valor")
liberacoes = read("liberacoes")
parcelas = read("parcelas")

assert contratos["idContrato"].is_unique
ids = set(contratos["idContrato"])

# A seleção de contratos guia tudo: só entram linhas dos contratos selecionados.
liberacoes = liberacoes[liberacoes["id_Contrato"].isin(ids)].copy()
parcelas = parcelas[parcelas["id_contrato"].isin(ids)].copy()
aditivos_prazo_sel = aditivos_prazo[aditivos_prazo["idContrato"].isin(ids)].copy()
aditivos_valor = aditivos_valor[aditivos_valor["idContrato"].isin(ids)].copy()

print("Pasta de dados:", base)
print({
    "contratos": len(contratos),
    "liberacoes": len(liberacoes),
    "parcelas": len(parcelas),
    "aditivos_prazo": len(aditivos_prazo_sel),
    "aditivos_valor": len(aditivos_valor),
})

Pasta de dados: /home/claude/work
{'contratos': 48, 'liberacoes': 886, 'parcelas': 897, 'aditivos_prazo': 85, 'aditivos_valor': 134}


## 2. Alvo com ajuste por aditivos de prazo

Fim original = `data_ordem_servico + prazo_total`. Com aditivo, o fim ajustado é a maior `Data_termino_nova` do contrato (término vigente). Essas colunas servem **só para construir o alvo**.

In [3]:
df = contratos.copy()

df["fim_original"] = df["data_ordem_servico"] + pd.to_timedelta(df["prazo_total"], unit="D")

fim_aditado = aditivos_prazo_sel.groupby("idContrato")["Data_termino_nova"].max().rename("fim_aditado")
df = df.merge(fim_aditado, on="idContrato", how="left")

df["fim_ajustado"] = df[["fim_original", "fim_aditado"]].max(axis=1)
df["prazo_ajustado"] = (df["fim_ajustado"] - df["data_ordem_servico"]).dt.days
df["duracao_real"] = (df["data_recebimento_provisorio"] - df["data_ordem_servico"]).dt.days

df["target_dias_atraso"] = (df["duracao_real"] - df["prazo_ajustado"]).clip(lower=0)

print("Contratos com atraso:", int((df["target_dias_atraso"] > 0).sum()), "de", len(df))

Contratos com atraso: 17 de 48


## 3. Colunas originais de cada fonte

Cada coluna de cada tabela é classificada como **feature** ou **descartada**, com o motivo, e uma checagem garante que nenhuma coluna ficou sem classificação. Não há nenhuma coluna calculada.

Critérios de descarte: chaves e identificadores; datas brutas (só virariam feature depois de derivações); texto livre; colunas constantes na seleção; colunas que repetem informação do contrato.

In [4]:
# (feature numérica, feature categórica, descartada com motivo) por fonte
CLASSIFICACAO = {
    "contratos": {
        "num": ["Perc_desconto"],
        "cat": ["Classificacao_contrato", "id_setor_divisao"],
        "descartadas": {
            "removida a pedido": ["prazo_total", "valor_total_contrato", "Valor_licitado", "Flag_ativo_emissao_OS"],
            "identificador": [
                "idContrato", "id_predio", "id_projeto_associado", "id_Licitacao", "id_fornecedor",
                "numero_contrato", "NumeroContrato", "processo_TRP", "processo_TRD",
                "SEI_contrato", "SEI_licitacao", "id_classificacao_contrato",
            ],
            "constante": [
                "id_diretoria", "Instrumento_contratual", "tipo_de_contrato", "id_tipo_contrato",
                "id_tipo_de_contrato", "Informacoes_para_relatorio_proj", "Possui_etapas_projeto",
                "Numero_etapas_projeto", "Pauta_GEP", "Status_contrato", "CT_paralisado",
                "CT_concluido", "CT_rescindido", "Data_paralisacao", "Motivo_paralisacao",
                "Data_previsao_retomada",
            ],
            "pós-obra (vazamento)": [
                "data_conclusao", "observacao_conclusao", "data_recebimento_provisorio",
                "data_assinatura_TRP", "data_recebimento_TRD", "data_assinatura_TRD", "observacao_contrato",
            ],
            "data bruta": ["data_assinatura", "data_proposta", "DataOrcamento", "data_ordem_servico", "data_ref_reajuste"],
            "texto livre / proxy de tempo": ["Descricao", "email_fornecedor_contrato", "ano_contrato"],
        },
    },
    "liberacoes": {
        "num": ["Valor_total_liberado", "Abono_atraso", "Atraso_considerado"],
        "cat": ["Status_liberacao"],
        "descartadas": {
            "chave": ["id_Contrato", "id_Parcela"],
            "data bruta": ["Data_efet_conclusao", "Data_liberação"],
        },
    },
    "parcelas": {
        "num": ["valor_reajustado"],
        "cat": ["tipo_de_parcela", "status_da_parcela"],
        "descartadas": {
            "chave": ["id_parcela", "id_contrato"],
            "data bruta": ["data_prevista_parcela", "data_efetiva", "data_liberacao"],
            "constante": ["ordem_status", "valor_ja_liberado"],
            "texto livre (311 rótulos)": ["nome_parcela"],
        },
    },
    "aditivos_prazo": {
        "num": ["Aditado_em_dias"],
        "cat": ["Status_aditivo_prazo", "Status_cadastro_parc_ta_prazo"],
        "descartadas": {
            "chave": ["idContrato"],
            "data bruta": ["Data_termino_anterior", "Data_termino_nova", "Data_assinatura_ta_prazo"],
            "repete o contrato": ["tipo_de_contrato", "Classificacao_contrato", "NumeroContrato"],
        },
    },
    "aditivos_valor": {
        "num": [],
        "cat": ["Status_aditivo_valor", "Status_cadastro_parc_ta_valor"],
        "descartadas": {
            "chave": ["idContrato"],
            "data bruta": ["Data_assinatura_TA_valor"],
            "constante": ["Nome_do_aditivo", "status_criacao"],
            "texto livre": ["Observacao_TA_valor"],
            "repete o contrato": ["tipo_de_contrato", "Classificacao_contrato", "NumeroContrato"],
        },
    },
}

TABELAS = {
    "contratos": contratos, "liberacoes": liberacoes, "parcelas": parcelas,
    "aditivos_prazo": aditivos_prazo_sel, "aditivos_valor": aditivos_valor,
}

linhas = []
for fonte, cfg in CLASSIFICACAO.items():
    tabela = TABELAS[fonte]
    descartadas = [c for cols in cfg["descartadas"].values() for c in cols]
    usadas = cfg["num"] + cfg["cat"]

    assert set(descartadas) | set(usadas) == set(tabela.columns), f"{fonte}: coluna sem classificação"
    assert not set(descartadas) & set(usadas), f"{fonte}: coluna em dois grupos"
    for c in cfg["descartadas"].get("constante", []):
        assert tabela[c].nunique(dropna=False) == 1, f"{fonte}.{c} não é constante"

    linhas.append({
        "fonte": fonte, "colunas_originais": len(tabela.columns),
        "features_numericas": ", ".join(cfg["num"]) or "-",
        "features_categoricas": ", ".join(cfg["cat"]) or "-",
        "descartadas": len(descartadas),
    })

display(pd.DataFrame(linhas))

,fonte,colunas_originais,features_numericas,features_categoricas,descartadas
0,contratos,50,Perc_desconto,"Classificacao_contrato, id_setor_divisao",47
1,liberacoes,8,"Valor_total_liberado, Abono_atraso, Atraso_considerado",Status_liberacao,4
2,parcelas,11,valor_reajustado,"tipo_de_parcela, status_da_parcela",8
3,aditivos_prazo,10,Aditado_em_dias,"Status_aditivo_prazo, Status_cadastro_parc_ta_prazo",7
4,aditivos_valor,10,-,"Status_aditivo_valor, Status_cadastro_parc_ta_valor",8


## 4. Montagem do dataset

- **Liberações + parcelas:** cada liberação recebe os campos originais da sua parcela (`tipo_de_parcela`, `status_da_parcela`, `valor_reajustado`). A chave é `id_Contrato` + `id_Parcela` contra `id_contrato` + `id_parcela`. Em `vw_bi_parcelas` o mesmo `id_parcela` se repete (execuções parciais), mas esses três campos são iguais nas repetições, então basta uma linha por `id_parcela`.
- **Aditivos de prazo e de valor:** ligados por `idContrato`.
- **Contratos sem registro** em uma fonte permanecem no dataset com os campos dessa fonte vazios (imputados na modelagem: 0 nas numéricas e `sem_registro` nas categóricas).

In [5]:
C = CLASSIFICACAO

# --- fonte 1: liberações + parcelas (grão: liberação)
parcelas_unicas = parcelas.drop_duplicates(["id_contrato", "id_parcela"])
assert parcelas_unicas.groupby(["id_contrato", "id_parcela"])[C["parcelas"]["num"] + C["parcelas"]["cat"]].nunique().max().max() == 1

lib_parc = liberacoes.merge(
    parcelas_unicas[["id_contrato", "id_parcela"] + C["parcelas"]["num"] + C["parcelas"]["cat"]],
    left_on=["id_Contrato", "id_Parcela"], right_on=["id_contrato", "id_parcela"], how="left",
)
assert len(lib_parc) == len(liberacoes), "o join de parcelas multiplicou linhas"
assert lib_parc["tipo_de_parcela"].notna().all(), "liberação sem parcela correspondente"

lib_parc = lib_parc.rename(columns={"id_Contrato": "idContrato"})[
    ["idContrato"] + C["liberacoes"]["num"] + C["liberacoes"]["cat"] + C["parcelas"]["num"] + C["parcelas"]["cat"]
]

# --- fonte 2 e 3: aditivos
adit_prazo = aditivos_prazo_sel[["idContrato"] + C["aditivos_prazo"]["num"] + C["aditivos_prazo"]["cat"]]
adit_valor = aditivos_valor[["idContrato"] + C["aditivos_valor"]["cat"]]

FONTES = {
    "liberacoes_parcelas": {
        "dados": lib_parc,
        "num": C["liberacoes"]["num"] + C["parcelas"]["num"],
        "cat": C["liberacoes"]["cat"] + C["parcelas"]["cat"],
    },
    "aditivos_prazo": {"dados": adit_prazo, "num": C["aditivos_prazo"]["num"], "cat": C["aditivos_prazo"]["cat"]},
    "aditivos_valor": {"dados": adit_valor, "num": C["aditivos_valor"]["num"], "cat": C["aditivos_valor"]["cat"]},
}

BASE_COLS = ["idContrato", "NumeroContrato", "data_recebimento_provisorio", "target_dias_atraso"]


def montar(nomes_fontes):
    """contratos (originais) + fontes escolhidas, ligadas por idContrato com left join."""
    frame = df[BASE_COLS + C["contratos"]["num"] + C["contratos"]["cat"]]
    num = list(C["contratos"]["num"])
    cat = list(C["contratos"]["cat"])

    for nome in nomes_fontes:
        f = FONTES[nome]
        frame = frame.merge(f["dados"], on="idContrato", how="left")
        num += f["num"]
        cat += f["cat"]

    # cada contrato pesa igual, independentemente de quantas linhas o join gerou
    frame = frame.copy()
    frame["n_linhas_contrato"] = frame.groupby("idContrato")["idContrato"].transform("size")
    frame["peso"] = 1.0 / frame["n_linhas_contrato"]

    assert frame["idContrato"].nunique() == len(df)
    return frame, num, cat


CENARIOS = {
    "contrato": [],
    "+ liberações e parcelas": ["liberacoes_parcelas"],
    "+ aditivos de prazo": ["aditivos_prazo"],
    "+ aditivos de valor": ["aditivos_valor"],
    "todas as fontes": ["liberacoes_parcelas", "aditivos_prazo", "aditivos_valor"],
}

resumo_cenarios = []
for nome, fontes in CENARIOS.items():
    frame, num, cat = montar(fontes)
    resumo_cenarios.append({
        "cenario": nome, "linhas": len(frame), "contratos": frame["idContrato"].nunique(),
        "features_numericas": len(num), "features_categoricas": len(cat),
        "linhas_por_contrato_max": int(frame["n_linhas_contrato"].max()),
    })
display(pd.DataFrame(resumo_cenarios))

dataset, NUM, CAT = montar(CENARIOS["todas as fontes"])
FEATURES = NUM + CAT
print("Dataset principal (todas as fontes):", dataset.shape)
display(dataset[["idContrato", "target_dias_atraso", "peso"] + FEATURES].head(8))

,cenario,linhas,contratos,features_numericas,features_categoricas,linhas_por_contrato_max
0,contrato,48,48,1,2,1
1,+ liberações e parcelas,887,48,5,5,110
2,+ aditivos de prazo,96,48,2,4,7
3,+ aditivos de valor,144,48,1,4,16
4,todas as fontes,18971,48,6,9,6804


Dataset principal (todas as fontes): (18971, 21)


,idContrato,target_dias_atraso,peso,Perc_desconto,Valor_total_liberado,Abono_atraso,Atraso_considerado,valor_reajustado,Aditado_em_dias,Classificacao_contrato,id_setor_divisao,Status_liberacao,tipo_de_parcela,status_da_parcela,Status_aditivo_prazo,Status_cadastro_parc_ta_prazo,Status_aditivo_valor,Status_cadastro_parc_ta_valor
0,11,25,0.056,0.000,"19,716.180",0.000,0.000,"19,716.180",NaN,Reforma Parcial,10,LIQUIDADO,Serviços Originais,LIQUIDADA,NaN,NaN,CONCLUÍDO,CADASTRADAS
1,11,25,0.056,0.000,"19,716.180",0.000,0.000,"19,716.180",NaN,Reforma Parcial,10,LIQUIDADO,Serviços Originais,LIQUIDADA,NaN,NaN,CONCLUÍDO,CADASTRADAS
2,11,25,0.056,0.000,"27,979.170",0.000,0.000,"27,979.170",NaN,Reforma Parcial,10,LIQUIDADO,Serviços Originais,LIQUIDADA,NaN,NaN,CONCLUÍDO,CADASTRADAS
3,11,25,0.056,0.000,"27,979.170",0.000,0.000,"27,979.170",NaN,Reforma Parcial,10,LIQUIDADO,Serviços Originais,LIQUIDADA,NaN,NaN,CONCLUÍDO,CADASTRADAS
4,11,25,0.056,0.000,"11,533.210",0.000,0.000,"11,533.210",NaN,Reforma Parcial,10,LIQUIDADO,Serviços Originais,LIQUIDADA,NaN,NaN,CONCLUÍDO,CADASTRADAS
5,11,25,0.056,0.000,"11,533.210",0.000,0.000,"11,533.210",NaN,Reforma Parcial,10,LIQUIDADO,Serviços Originais,LIQUIDADA,NaN,NaN,CONCLUÍDO,CADASTRADAS
6,11,25,0.056,0.000,"165,393.570",0.000,0.000,"165,393.570",NaN,Reforma Parcial,10,LIQUIDADO,Serviços Originais,LIQUIDADA,NaN,NaN,CONCLUÍDO,CADASTRADAS
7,11,25,0.056,0.000,"165,393.570",0.000,0.000,"165,393.570",NaN,Reforma Parcial,10,LIQUIDADO,Serviços Originais,LIQUIDADA,NaN,NaN,CONCLUÍDO,CADASTRADAS


## 5. Separação desenvolvimento × teste final (por contrato)

In [6]:
ordem = (
    df.sort_values(["data_recebimento_provisorio", "idContrato"])["idContrato"].tolist()
)
dev_ids, test_ids = ordem[:-N_TEST], ordem[-N_TEST:]

fim = df.set_index("idContrato")["data_recebimento_provisorio"]
assert fim[dev_ids].max() <= fim[test_ids].min()

folds = []
for k in range(N_FOLDS):
    n_train = MIN_TRAIN + k * BLOCK
    folds.append({"fold": k + 1, "train": dev_ids[:n_train], "val": dev_ids[n_train:n_train + BLOCK]})

print("Contratos de teste:", test_ids)
display(pd.DataFrame([{"fold": f["fold"], "contratos_treino": len(f["train"]), "contratos_validacao": len(f["val"])} for f in folds]))

Contratos de teste: [173, 201, 287, 283, 342, 317, 254, 377]


,fold,contratos_treino,contratos_validacao
0,1,20,5
1,2,25,5
2,3,30,5
3,4,35,5


## 6. Modelos e métricas

- **Baseline:** mediana (ponderada) do atraso no treino.
- **Ridge:** numéricas com imputação por 0 e padronização; categóricas com `sem_registro` para vazios e one-hot. As previsões das linhas são médias por contrato e depois truncadas em 0.

In [7]:
def make_ridge(num, cat):
    transformers = [("num", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value=0)), ("sc", StandardScaler())]), num)]
    transformers.append((
        "cat",
        Pipeline([
            ("imp", SimpleImputer(strategy="constant", fill_value="sem_registro")),
            ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]),
        cat,
    ))
    return Pipeline([("prep", ColumnTransformer(transformers)), ("model", Ridge(alpha=RIDGE_ALPHA))])


def fit_predict(frame, num, cat, train_ids, eval_ids):
    cols = num + cat
    tr = frame[frame["idContrato"].isin(train_ids)]
    te = frame[frame["idContrato"].isin(eval_ids)]

    baseline = DummyRegressor(strategy="median").fit(tr[cols], tr["target_dias_atraso"], sample_weight=tr["peso"])
    ridge = make_ridge(num, cat)
    ridge.fit(tr[cols], tr["target_dias_atraso"], model__sample_weight=tr["peso"])

    linhas = te[["idContrato", "target_dias_atraso"]].assign(
        baseline_mediana=baseline.predict(te[cols]), ridge=ridge.predict(te[cols])
    )
    por_contrato = linhas.groupby("idContrato").agg(
        real=("target_dias_atraso", "first"), baseline_mediana=("baseline_mediana", "mean"), ridge=("ridge", "mean")
    )
    por_contrato[["baseline_mediana", "ridge"]] = por_contrato[["baseline_mediana", "ridge"]].clip(lower=0)
    return por_contrato.reset_index(), ridge


def metrics(y_true, y_pred, full=False):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    atrasados = y_true > 0

    out = {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "mae_so_atrasados": mean_absolute_error(y_true[atrasados], y_pred[atrasados]) if atrasados.any() else np.nan,
        "vies_medio": (y_pred - y_true).mean(),
    }
    if full:
        out["r2"] = r2_score(y_true, y_pred)
        out["spearman"] = spearmanr(y_true, y_pred)[0] if np.unique(y_pred).size > 1 else np.nan
    return out


def run_cv(frame, num, cat):
    cv_rows, oof_rows = [], []
    for f in folds:
        pred, _ = fit_predict(frame, num, cat, f["train"], f["val"])
        for nome in ["baseline_mediana", "ridge"]:
            cv_rows.append({"fold": f["fold"], "modelo": nome, **metrics(pred["real"], pred[nome])})
        oof_rows.append(pred.assign(fold=f["fold"]))
    return pd.DataFrame(cv_rows), pd.concat(oof_rows, ignore_index=True)

## 7. Validação temporal — dataset com todas as fontes

In [8]:
cv, oof = run_cv(dataset, NUM, CAT)

print("MAE por janela:")
display(cv.pivot(index="fold", columns="modelo", values="mae"))

print("Resumo entre janelas:")
cv_resumo = cv.groupby("modelo").agg(mae_medio=("mae", "mean"), mae_dp=("mae", "std"), rmse_medio=("rmse", "mean"))
display(cv_resumo)

print("Métricas agregadas (20 contratos de validação):")
display(pd.DataFrame({n: metrics(oof["real"], oof[n], full=True) for n in ["baseline_mediana", "ridge"]}).T)

MAE por janela:


modelo,baseline_mediana,ridge
fold,,
1,30.000,21.819
2,63.800,68.437
3,0.000,38.394
4,17.800,118.114


Resumo entre janelas:


,mae_medio,mae_dp,rmse_medio
modelo,,,
baseline_mediana,27.900,26.917,52.053
ridge,61.691,42.275,88.765


Métricas agregadas (20 contratos de validação):


,mae,rmse,mae_so_atrasados,vies_medio,r2,spearman
baseline_mediana,27.900,69.787,79.714,-27.900,-0.190,NaN
ridge,61.691,101.652,84.736,20.871,-1.525,0.051


## 8. Teste final

In [9]:
test_pred, ridge_final = fit_predict(dataset, NUM, CAT, dev_ids, test_ids)

display(pd.DataFrame({n: metrics(test_pred["real"], test_pred[n], full=True) for n in ["baseline_mediana", "ridge"]}).T)
display(test_pred)

coef = pd.Series(
    ridge_final["model"].coef_, index=ridge_final["prep"].get_feature_names_out()
).sort_values(key=np.abs, ascending=False)
print("Maiores coeficientes (variáveis padronizadas):")
display(coef.head(10).to_frame("coeficiente"))

,mae,rmse,mae_so_atrasados,vies_medio,r2,spearman
baseline_mediana,21.875,54.352,87.500,-21.875,-0.193,NaN
ridge,48.581,62.013,45.722,27.634,-0.553,0.156


,idContrato,real,baseline_mediana,ridge
0,173,0,0.000,95.445
1,201,0,0.000,0.000
2,254,0,0.000,15.905
3,283,0,0.000,59.006
4,287,0,0.000,100.821
5,317,23,0.000,30.657
6,342,0,0.000,26.024
7,377,152,0.000,68.214


Maiores coeficientes (variáveis padronizadas):


,coeficiente
cat__id_setor_divisao_12,-35.384
cat__Classificacao_contrato_Construção,-22.858
cat__id_setor_divisao_10,22.086
num__Perc_desconto,18.666
cat__status_da_parcela_sem_registro,17.398
cat__tipo_de_parcela_sem_registro,17.398
cat__Status_liberacao_sem_registro,17.398
cat__Status_liberacao_LIQUIDADO,-17.398
cat__status_da_parcela_LIQUIDADA,-17.398
cat__id_setor_divisao_8,17.342


## 9. Efeito de cada fonte (apenas validação temporal)

Compara os cenários da seção 4, cada um com seu próprio dataset. O teste final não é usado aqui.

In [10]:
rows = []
for nome, fontes in CENARIOS.items():
    frame, num, cat = montar(fontes)
    cv_c, _ = run_cv(frame, num, cat)
    mae = cv_c.groupby("modelo")["mae"].mean()
    rows.append({
        "dados": nome,
        "linhas": len(frame),
        "n_features": len(num) + len(cat),
        "mae_baseline": mae["baseline_mediana"],
        "mae_ridge": mae["ridge"],
        "ridge_menos_baseline": mae["ridge"] - mae["baseline_mediana"],
    })

display(pd.DataFrame(rows))

,dados,linhas,n_features,mae_baseline,mae_ridge,ridge_menos_baseline
0,contrato,48,3,27.900,65.669,37.769
1,+ liberações e parcelas,887,10,27.900,63.250,35.350
2,+ aditivos de prazo,96,6,27.900,68.797,40.897
3,+ aditivos de valor,144,5,27.900,65.970,38.070
4,todas as fontes,18971,15,27.900,61.691,33.791


## 10. Exportação do dataset de treino

In [11]:
OUTPUT_DIR = Path("08_dataset_features_originais_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

saida = dataset[["idContrato", "NumeroContrato", "target_dias_atraso", "peso"] + FEATURES]
saida.sort_values("idContrato").to_csv(OUTPUT_DIR / "dataset_treino_features_originais.csv", index=False, encoding="utf-8-sig")

print("Salvo em:", (OUTPUT_DIR / "dataset_treino_features_originais.csv").resolve(), "|", saida.shape)

Salvo em: /home/claude/work/08_dataset_features_originais_outputs/dataset_treino_features_originais.csv | (18971, 19)


## Leitura dos resultados

**Com colunas apenas originais, o Ridge também não supera o baseline.**

| | MAE validação temporal (média de 4 janelas) | MAE teste final (8 contratos) |
|---|---|---|
| Baseline (mediana = 0 dia) | 27,9 | 21,9 |
| Ridge, todas as fontes (15 features originais) | 61,7 | 48,6 |
| Ridge, só colunas do contrato (notebook 06) | 65,7 | 40,8 |
| Ridge, features agregadas (notebook 07) | 52,6 | 60,6 |

- **Ganho pequeno com as fontes:** na seção 9, cada fonte reduz ou aumenta o MAE do Ridge em poucos dias em relação ao cenário só com o contrato (65,7), e o melhor cenário (todas as fontes) chega a 61,7. Todos ficam muito acima do baseline.
- **Pouco sinal nas colunas originais:** a maioria é status quase constante (`Status_liberacao` é LIQUIDADO em 883 de 886 linhas; os status de aditivo são CONCLUÍDO em quase todos) ou valores de uma única parcela ou liberação. A informação que as tabelas trazem sobre o contrato (quantas parcelas, quantos aditivos, quanto foi liberado) só aparece quando as linhas são somadas ou contadas, o que o 07 fazia e este notebook não faz.
- **Ordenação fraca:** o Spearman é 0,05 na validação e 0,16 no teste (0,28 e 0,52 no notebook 07). Com 8 contratos e 2 atrasados, o valor do teste não é conclusivo.
- **Indicadores `sem_registro`:** os coeficientes de `tipo_de_parcela`, `status_da_parcela` e `Status_liberacao` com valor `sem_registro` são idênticos, porque marcam o mesmo contrato (o 12, único sem liberações nem parcelas). O efeito dessas colunas é, na prática, memorizar o atraso desse contrato (139 dias).
- **Multiplicação de linhas:** o join completo gera 18.971 linhas para 48 contratos (até 6.804 linhas em um único contrato). O peso `1 / nº de linhas` equilibra o treino, mas as linhas de um mesmo contrato são combinações repetidas dos mesmos registros, então o dataset tem muito menos informação do que o número de linhas sugere.
- **Janelas instáveis:** o erro do Ridge varia de 22 (janela 1) a 118 (janela 4), com desvio-padrão de 42, contra 27 do baseline.
- **Vazamento:** `Atraso_considerado`, `Abono_atraso` e `Aditado_em_dias` só existem durante ou depois da obra, e mesmo assim não bastam para superar o baseline.

**Ideias para testar depois (não executadas):** aumentar `RIDGE_ALPHA` ou usar `RidgeCV`; voltar às contagens e somas por contrato do notebook 07, que concentram a informação; ou cortar os dados da obra em um ponto fixo do tempo (como nos snapshots do 04B) para obter uma previsão utilizável durante a execução.